## Treinamento da rede e avaliacao para os dados de teste em Python (usando entrada_rede gerada em R)

In [5]:
# Configurações do ambiente em Python

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 2003
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.0025

np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
# Carregamento dos dados

entrada_rede = pd.read_csv("C:/Users/gusta/repositorio_tcc/tcc/pseudo-sobrevivencia-total/dados/entrada/entrada_rede.csv")

# Split treino/teste
dados_treino = entrada_rede.loc[entrada_rede["SET"] == "train"].copy()
dados_teste = entrada_rede.loc[entrada_rede["SET"] == "test"].copy()

In [7]:
dados_treino.head(20)

,ID,TRT,AGE,SEX,TIME,PSEUDO,SET
0,1,1,-11.035944,0,0.130140,1.000000,train
1,1,1,-11.035944,0,0.194520,1.000000,train
2,1,1,-11.035944,0,0.294520,1.000000,train
3,1,1,-11.035944,0,0.416440,1.000000,train
4,1,1,-11.035944,0,0.550680,1.000000,train
5,1,1,-11.035944,0,0.863010,1.000439,train
6,1,1,-11.035944,0,1.120545,1.001036,train
7,1,1,-11.035944,0,1.720550,-0.003955,train
8,1,1,-11.035944,0,2.798630,-0.003348,train
9,1,1,-11.035944,0,3.652738,-0.003023,train


In [8]:
dados_teste.head(20)

,ID,TRT,AGE,SEX,TIME,PSEUDO,SET
80,9,0,-14.082544,0,0.130140,1.000000e+00,test
81,9,0,-14.082544,0,0.194520,1.000000e+00,test
82,9,0,-14.082544,0,0.294520,1.000000e+00,test
83,9,0,-14.082544,0,0.416440,1.000000e+00,test
84,9,0,-14.082544,0,0.550680,1.000000e+00,test
85,9,0,-14.082544,0,0.863010,1.000439e+00,test
86,9,0,-14.082544,0,1.120545,1.001036e+00,test
87,9,0,-14.082544,0,1.720550,1.001663e+00,test
88,9,0,-14.082544,0,2.798630,1.005110e+00,test
89,9,0,-14.082544,0,3.652738,1.008500e+00,test


In [9]:
# Carregamento dos dados brutos para obter tempos de evento do teste
melanoma = pd.read_csv("C:/Users/gusta/repositorio_tcc/tcc/pseudo-sobrevivencia-total/dados/entrada/e1684_bruto.csv")

# IDs únicos do teste e tempos de evento correspondentes
ids_teste = np.sort(dados_teste["ID"].astype(int).unique())
e1684_teste = melanoma.loc[melanoma["ID"].isin(ids_teste), ["ID", "FAILTIME", "FAILCENS"]].copy()
tempos_eventos_teste = np.sort(e1684_teste.loc[e1684_teste["FAILCENS"] == 1, "FAILTIME"].unique())

print(f"Lista tempos de evento (teste): {tempos_eventos_teste}")

Lista tempos de evento (teste): [0.09863 0.12329 0.13151 0.1589  0.2411  0.28767 0.32055 0.34795 0.41096
 0.4137  0.43562 0.43836 0.55068 0.72603 0.89589 0.90137 0.92877 0.9863
 1.00274 1.09041 1.19178 1.52055 1.84658 1.85753 1.88219 2.72329 3.05479
 3.33973 3.86575 4.29041 4.45205]


In [10]:
def gera_matriz_design_one_hot(dados, tempos, media_idade, desvio_idade):
    base = pd.DataFrame({
        "TRT": dados["TRT"].astype(float).values,
        "AGE_NORM": ((dados["AGE"].astype(float).values - media_idade) / (desvio_idade + 1e-8)),
        "SEX": dados["SEX"].astype(float).values,
        "TIME": dados["TIME"].astype(float).values,
    })

    tcat = pd.Categorical(base["TIME"], categories=tempos, ordered=True)
    t_oh = pd.get_dummies(tcat, prefix="TIME")
    return pd.concat([base[["TRT", "AGE_NORM", "SEX"]], t_oh], axis=1)

media_idade = dados_treino["AGE"].astype(float).mean()
desvio_idade = dados_treino["AGE"].astype(float).std(ddof=0)
time_levels = np.sort(dados_treino["TIME"].astype(float).unique())

x_treino_onehot = gera_matriz_design_one_hot(dados_treino, time_levels, media_idade, desvio_idade)
y_treino_onehot = dados_treino["PSEUDO"].astype(float).values.reshape(-1, 1)

X_train_onehot = torch.tensor(x_treino_onehot.values.astype(np.float32))
y_train_onehot = torch.tensor(y_treino_onehot.astype(np.float32))

train_loader_onehot = DataLoader(
    TensorDataset(X_train_onehot, y_train_onehot),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)

print(f"dimensao X_train: {tuple(X_train_onehot.shape)}")
print(f"dimensao y_train: {tuple(y_train_onehot.shape)}")

dimensao X_train: (2270, 13)
dimensao y_train: (2270, 1)


In [11]:
# One hot
x_treino_onehot.head(20)

,TRT,AGE_NORM,SEX,TIME_0.13014,TIME_0.19452,TIME_0.29452,TIME_0.41644,TIME_0.55068,TIME_0.86301,TIME_1.120545,TIME_1.72055,TIME_2.79863,TIME_3.6527375
0,1.0,-0.866136,0.0,True,False,False,False,False,False,False,False,False,False
1,1.0,-0.866136,0.0,False,True,False,False,False,False,False,False,False,False
2,1.0,-0.866136,0.0,False,False,True,False,False,False,False,False,False,False
3,1.0,-0.866136,0.0,False,False,False,True,False,False,False,False,False,False
4,1.0,-0.866136,0.0,False,False,False,False,True,False,False,False,False,False
5,1.0,-0.866136,0.0,False,False,False,False,False,True,False,False,False,False
6,1.0,-0.866136,0.0,False,False,False,False,False,False,True,False,False,False
7,1.0,-0.866136,0.0,False,False,False,False,False,False,False,True,False,False
8,1.0,-0.866136,0.0,False,False,False,False,False,False,False,False,True,False
9,1.0,-0.866136,0.0,False,False,False,False,False,False,False,False,False,True


In [12]:
# Definição da rede neural

class rede_sobrevivencia(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, 2 * input_dim),
            nn.Tanh(),
            nn.Linear(2 * input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

model_onehot = rede_sobrevivencia(input_dim=X_train_onehot.shape[1])
criterion_onehot = nn.MSELoss()
optimizer_onehot = optim.Adam(model_onehot.parameters(), lr=LEARNING_RATE)

In [13]:
# Treinamento da rede

for epoch in range(EPOCHS):
    model_onehot.train()
    running_loss = 0.0

    for xb, yb in train_loader_onehot:
        optimizer_onehot.zero_grad()
        pred = model_onehot(xb)
        loss = criterion_onehot(pred, yb)
        loss.backward()
        optimizer_onehot.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Época {epoch + 1:4d}/{EPOCHS} | mse={(running_loss / len(train_loader_onehot)):.6f}")

Época   10/100 | mse=0.196175
Época   20/100 | mse=0.196192
Época   30/100 | mse=0.195293
Época   40/100 | mse=0.192320
Época   50/100 | mse=0.193443
Época   60/100 | mse=0.192516
Época   70/100 | mse=0.191063
Época   80/100 | mse=0.191526
Época   90/100 | mse=0.191195
Época  100/100 | mse=0.191578


In [14]:
# Pasta de saída das predições
dir_saida = "C:/Users/gusta/repositorio_tcc/tcc/pseudo-sobrevivencia-total/dados/saida"

# IDs únicos de teste
dados_ids_teste = dados_teste[["ID", "TRT", "AGE", "SEX"]].drop_duplicates().sort_values("ID")

# Base com os tempos de evento do teste para avaliação
linhas_teste_evento = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_evento in tempos_eventos_teste:
        linhas_teste_evento.append({
            "ID": int(linha_individuo["ID"]),
            "TRT": float(linha_individuo["TRT"]),
            "AGE": float(linha_individuo["AGE"]),
            "SEX": float(linha_individuo["SEX"]),
            "TIME": float(tempo_evento),
        })
dados_teste_evento = pd.DataFrame(linhas_teste_evento)

# Previsões Rede one-hot
linhas_grade_A = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_grade in time_levels:
        linhas_grade_A.append({
            "ID": int(linha_individuo["ID"]),
            "TRT": float(linha_individuo["TRT"]),
            "AGE": float(linha_individuo["AGE"]),
            "SEX": float(linha_individuo["SEX"]),
            "TIME": float(tempo_grade),
        })

grade_predicao_A = pd.DataFrame(linhas_grade_A)
x_pred_A = gera_matriz_design_one_hot(grade_predicao_A, time_levels, media_idade, desvio_idade)
x_pred_A = x_pred_A.reindex(columns=x_treino_onehot.columns, fill_value=0.0)

# Predição da Rede one-hot na grade de treino
model_onehot.eval()
with torch.no_grad():
    grade_predicao_A["PRED_S"] = model_onehot(torch.tensor(x_pred_A.values.astype(np.float32))).cpu().numpy().reshape(-1)

grade_predicao_A = grade_predicao_A.sort_values(["ID", "TIME"]).reset_index(drop=True)

# Predição da Rede one-hot nos tempos de evento do teste (via interpolação)
linhas_evento_A = []
for id_individuo, predicoes_individuo in grade_predicao_A.groupby("ID"):
    predicoes_individuo = predicoes_individuo.sort_values("TIME")
    predicoes_evento = np.interp(
        tempos_eventos_teste,
        predicoes_individuo["TIME"].values,
        predicoes_individuo["PRED_S"].values,
    )
    for tempo_evento, valor_predito in zip(tempos_eventos_teste, predicoes_evento):
        linhas_evento_A.append({
            "ID": int(id_individuo),
            "TIME": float(tempo_evento),
            "PRED_S": float(valor_predito),
        })


pred_event_A = pd.DataFrame(linhas_evento_A).sort_values(["ID", "TIME"]).reset_index(drop=True)



arquivo_saida_A = f"{dir_saida}/predicao-rede-onehot.csv"
pred_event_A.to_csv(arquivo_saida_A, index=False)

In [ ]:
# Função para calcular o CTD
def calc_ctd(outcomes_df, pred_event_df):

    surv_at_event = pred_event_df.pivot(index="ID", columns="TIME", values="PRED_S")
    outcomes = outcomes_df[["ID", "FAILTIME", "FAILCENS"]].drop_duplicates("ID").copy()
    outcomes = outcomes.sort_values("FAILTIME").reset_index(drop=True)

    concordante = 0
    empates = 0
    pares_comparaveis = 0

    for i in range(len(outcomes)):
        if int(outcomes.loc[i, "FAILCENS"]) != 1:
            continue

        id_individuo_I = int(outcomes.loc[i, "ID"])
        tempo_individuo_I = float(outcomes.loc[i, "FAILTIME"])

        if id_individuo_I not in surv_at_event.index or tempo_individuo_I not in surv_at_event.columns:
            print(f"Indivíduo {id_individuo_I} não encontrado nas predições, pulando comparação.")
            continue

        risck_at_event_I = 1.0 - float(surv_at_event.loc[id_individuo_I, tempo_individuo_I])
        js_comparaveis = outcomes.index[outcomes["FAILTIME"] > tempo_individuo_I].tolist()

        for j in js_comparaveis:
            id_individuo_J = int(outcomes.loc[j, "ID"])
            if id_individuo_J not in surv_at_event.index:
                print(f"Indivíduo {id_individuo_J} não encontrado nas predições, pulando comparação.")
                continue

            risk_at_event_J = 1.0 - float(surv_at_event.loc[id_individuo_J, tempo_individuo_I])
            if np.isnan(risck_at_event_I) or np.isnan(risk_at_event_J):
                print(f"Risco NaN para indivíduos {id_individuo_I} ou {id_individuo_J}, pulando comparação.")
                print(f"Risco risck_at_event_I: {risck_at_event_I}, risk_at_event_J: {risk_at_event_J}")
                continue

            pares_comparaveis += 1
            if risck_at_event_I > risk_at_event_J:
                concordante += 1
            elif risck_at_event_I == risk_at_event_J:
                empates += 1

    ctd = (concordante + 0.5 * empates) / pares_comparaveis if pares_comparaveis > 0 else np.nan
    return ctd, pares_comparaveis, concordante, empates

# Monta DF sem extrapolação dos limites de tempo em que a rede foi treinada
def filtrar_dados_sem_extrapolacao(outcomes_df , pred_event_df, time_levels):
    t_min = float(np.min(time_levels))
    t_max = float(np.max(time_levels))

    # Filtra os dados reais (eventos)
    out_test_sem_extrap = outcomes_df.loc[
        (outcomes_df["FAILTIME"] >= t_min) & (outcomes_df["FAILTIME"] <= t_max)
    ].copy()

    # Filtra as predições
    pred_event_sem_extrap = pred_event_df.loc[
        (pred_event_df["TIME"] >= t_min) & (pred_event_df["TIME"] <= t_max)
    ].copy()

    return out_test_sem_extrap, pred_event_sem_extrap


# Resultados Obtidos

### Rede Neural A: one-hot encoding

In [20]:
# Utiliza as predições da rede one-hot
pred_event_A

out_test_sem_extrapA, pred_event_sem_extrapA = filtrar_dados_sem_extrapolacao(
    e1684_teste, 
    pred_event_A, 
    time_levels)


ctd_sem_extrapA, comp_sem_extrapA, conc_sem_extrapA, emaptes_sem_extrapA = calc_ctd(
    out_test_sem_extrapA,
    pred_event_sem_extrapA,
)

# CTD com todos os eventos (inclui extrapolação)
ctd_todosA, comp_todosA, conc_todosA, ties_todosA = calc_ctd(e1684_teste, pred_event_A)

avaliacao_ctd = pd.DataFrame([
    {
        "CENARIO": "sem_extrapolacao",
        "CTD_TESTE": ctd_sem_extrapA,
        "COMPARAVEIS": comp_sem_extrapA,
        "CONCORDANTES": conc_sem_extrapA,
        "EMPATES": emaptes_sem_extrapA,
    },
    {
        "CENARIO": "todos_eventos",
        "CTD_TESTE": ctd_todosA,
        "COMPARAVEIS": comp_todosA,
        "CONCORDANTES": conc_todosA,
        "EMPATES": ties_todosA,
    },
])

print(avaliacao_ctd)

            CENARIO  CTD_TESTE  COMPARAVEIS  CONCORDANTES  EMPATES
0  sem_extrapolacao   0.606061          429           260        0
1     todos_eventos   0.481259         1334           642        0
